# Predictive Maintenance of Industrial Machinery
### AICTE Project – Problem Statement #39

**End-to-end notebook:** EDA → Feature Engineering → Model Training → Evaluation → IBM Cloud Deployment

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import config
from src.data_loader import load_dataset
from src.preprocessing import clean_data, engineer_features, encode_target, split_data, apply_smote, LABEL_INVERSE_MAP
from src.model_training import train_and_select, save_model
from src.evaluation import evaluate_model, plot_confusion_matrix, plot_feature_importance, plot_cv_results
from src.predict import predict_batch

print('All imports successful ✓')

## 1. Load & Explore Dataset

In [ ]:
df = load_dataset()
print('Shape:', df.shape)
df.head()

In [ ]:
print('Failure Type distribution:')
print(df['Failure Type'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['Failure Type'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Failure Type Distribution')
axes[0].set_xlabel('Failure Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

df['Type'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#3b82d4','#7c5cd8','#e5a020'])
axes[1].set_title('Machine Type Distribution')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric features
numeric_cols = ['Air temperature [K]', 'Process temperature [K]',
                'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure']
existing = [c for c in numeric_cols if c in df.columns]

plt.figure(figsize=(8, 6))
sns.heatmap(df[existing].corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 2. Preprocessing & Feature Engineering

In [ ]:
df_clean = clean_data(df)
df_feat  = engineer_features(df_clean)
X, y     = encode_target(df_feat)

print('Features shape:', X.shape)
print('Class distribution (after encoding):')
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {LABEL_INVERSE_MAP[u]}: {c}')

## 3. Train / Test Split + SMOTE

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)
X_train_bal, y_train_bal = apply_smote(X_train.values, y_train)
X_train_bal = pd.DataFrame(X_train_bal, columns=X_train.columns)
print('Balanced training set shape:', X_train_bal.shape)

## 4. Model Training & Cross-Validation

In [ ]:
numeric_features = [c for c in config.NUMERIC_FEATURES if c in X.columns]
engineered = ['power_W','temp_delta_K','tool_torque_product','speed_torque_ratio']
all_numeric = numeric_features + [f for f in engineered if f in X.columns]
categorical_features = [c for c in config.CATEGORICAL_FEATURES if c in X.columns]

best_pipeline, cv_results = train_and_select(
    X_train_bal, y_train_bal,
    numeric_features=all_numeric,
    categorical_features=categorical_features,
)
print('CV Results:', cv_results)

In [ ]:
plot_cv_results(cv_results)
plt.show()

## 5. Evaluation

In [ ]:
metrics = evaluate_model(best_pipeline, X_test, y_test)
print(f"Accuracy  : {metrics['accuracy']:.4f}")
print(f"F1 (macro): {metrics['f1_macro']:.4f}")
print(f"F1 (wtd)  : {metrics['f1_weighted']:.4f}")

In [ ]:
plot_confusion_matrix(best_pipeline, X_test, y_test, save=False)
plt.show()

In [ ]:
all_feature_names = all_numeric + categorical_features
plot_feature_importance(best_pipeline, all_feature_names, save=False)
plt.show()

## 6. Save Model & IBM Cloud Deployment

In [ ]:
model_path = save_model(best_pipeline)
print('Model saved:', model_path)

In [ ]:
# IBM Cloud – uncomment after filling .env credentials
# from src.ibm_cloud import get_cos_client, upload_file_to_cos, get_wml_client, store_model_in_wml, deploy_model_in_wml

# cos = get_cos_client()
# upload_file_to_cos(model_path, cos_client=cos)

# wml = get_wml_client()
# model_uid = store_model_in_wml(best_pipeline, wml_client=wml)
# deployment_uid = deploy_model_in_wml(model_uid, wml_client=wml)
# print('Deployment UID:', deployment_uid)

## 7. Real-time Inference Demo

In [ ]:
demo_samples = [
    {'Type': 'M', 'Air temperature [K]': 298.1, 'Process temperature [K]': 308.6,
     'Rotational speed [rpm]': 1551, 'Torque [Nm]': 42.8, 'Tool wear [min]': 0},
    {'Type': 'L', 'Air temperature [K]': 302.5, 'Process temperature [K]': 312.2,
     'Rotational speed [rpm]': 1380, 'Torque [Nm]': 55.0, 'Tool wear [min]': 220},
    {'Type': 'H', 'Air temperature [K]': 305.0, 'Process temperature [K]': 311.5,
     'Rotational speed [rpm]': 1200, 'Torque [Nm]': 30.0, 'Tool wear [min]': 50},
]

results = predict_batch(best_pipeline, demo_samples)
results[['predicted_failure_type']].assign(sample=range(1, len(results)+1)).set_index('sample')